# Notebook 01: Generate DPO Preference Pairs

**Purpose:** Build ~150 DAA preference pairs where `chosen` = Socratic guidance, `rejected` = answer-dump.

**Strategy:** Since you're generating manually via Claude.ai web UI, this notebook helps you do it efficiently in batches. We use ~150 pairs (not 500) because:
- Behavioral steering (not knowledge addition) needs less data
- 100-200 pairs is well-established for DPO behavioral shifts on small models
- 150 pairs ≈ 8-10 Claude.ai conversations of batched requests = ~1-2 hours of work

**Output:** `data/dpo_pairs.json` - the preference dataset for DPO training.

---

## Workflow Overview

1. We generate 150 DAA question stubs in code (covers complexity, recurrences, algorithms, data structures)
2. You take batches of 10 questions to Claude.ai, asking it to write both versions for each
3. You paste responses back into a structured format
4. We compile the final `dpo_pairs.json`


## Step 1: Setup

In [ ]:
import sys
import json
import random
from pathlib import Path

KAGGLE = Path("/kaggle").exists()
PROJECT_ROOT = Path("/kaggle/working/daa-helper") if KAGGLE else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

from utils.io_helpers import save_json, load_json

random.seed(42)


## Step 2: Generate 150 DAA question stubs across categories

In [ ]:
# We programmatically generate question templates to ensure diversity.
# These become the prompts for which we'll create chosen/rejected pairs.

DAA_QUESTION_TEMPLATES = {
    "complexity_analysis": [
        "What is the time complexity of a nested loop that iterates n times outer and {inner} times inner?",
        "Analyze the time complexity of {algo}.",
        "What is the space complexity of {algo}?",
        "How does the time complexity of {algo} compare on best vs worst case?",
        "Walk me through the complexity of this code: a single loop from 1 to n that does {op} inside.",
    ],
    "recurrence_relations": [
        "Solve T(n) = {a}T(n/{b}) + n^{k} using the Master Theorem.",
        "Use the recursion tree method to solve T(n) = {a}T(n/{b}) + {fn}.",
        "Solve T(n) = T(n-1) + {fn} by unrolling.",
        "Use substitution to verify T(n) = {a}T(n/{b}) + n is O(n log n).",
        "What is the recurrence for {algo} and how would you solve it?",
    ],
    "algorithm_comparison": [
        "Compare {algo1} and {algo2} for the problem of {problem}. When would you use each?",
        "Why is {algo1} preferred over {algo2} when {condition}?",
        "Trade-offs between {algo1} and {algo2}: discuss time, space, and use cases.",
    ],
    "data_structures": [
        "Why is the average insert into a hash table O(1) but worst case O(n)?",
        "Explain the time complexity of all operations on a {ds}.",
        "When should I use a {ds1} vs a {ds2}?",
        "How does a balanced BST maintain O(log n) operations?",
        "What is the amortized complexity of operations on a {ds}?",
    ],
    "algorithm_design": [
        "How would you approach the {problem} using {paradigm}?",
        "Why does a greedy approach fail for {problem}?",
        "Design a divide-and-conquer algorithm for {problem}.",
        "When does dynamic programming work, and how would I recognize it?",
        "What is the difference between top-down memoization and bottom-up DP?",
    ],
    "proof_correctness": [
        "How would you prove the correctness of {algo}?",
        "What is a loop invariant and how do I use one to prove {algo} correct?",
        "Why does the greedy choice property guarantee optimality for {problem}?",
    ],
    "lower_bounds": [
        "Why is comparison-based sorting bounded by Omega(n log n)?",
        "What is the lower bound for {problem} and why?",
        "Can {problem} be solved faster than O({bound})? Why or why not?",
    ]
}

FILLERS = {
    "inner": ["log n", "sqrt(n)", "n/2", "n"],
    "algo": ["merge sort", "quicksort", "binary search", "BFS", "DFS", "Dijkstra's algorithm",
             "heap sort", "insertion sort", "Floyd-Warshall", "Bellman-Ford", "Kruskal's algorithm",
             "Prim's algorithm", "Kadane's algorithm", "KMP string matching"],
    "algo1": ["merge sort", "quicksort", "heap sort", "Dijkstra", "BFS"],
    "algo2": ["bubble sort", "selection sort", "insertion sort", "Bellman-Ford", "DFS"],
    "op": ["a constant-time check", "a binary search", "another loop of size n"],
    "a": [2, 3, 4, 8],
    "b": [2, 3, 4],
    "k": [0, 1, 2, 3],
    "fn": ["n", "n log n", "1", "n^2", "log n"],
    "problem": ["shortest path", "minimum spanning tree", "longest common subsequence",
                "knapsack", "edit distance", "matrix chain multiplication", "coin change",
                "activity selection", "Huffman coding", "interval scheduling"],
    "condition": ["the data is nearly sorted", "memory is limited", "the input is small",
                  "we need guaranteed worst-case performance"],
    "ds": ["stack", "queue", "linked list", "hash table", "binary heap", "binary search tree",
           "AVL tree", "trie", "graph", "dynamic array"],
    "ds1": ["array", "linked list", "hash table", "BST"],
    "ds2": ["linked list", "array", "BST", "hash table"],
    "paradigm": ["dynamic programming", "greedy", "divide and conquer", "backtracking"],
    "bound": ["n", "n log n", "n^2"]
}


def fill_template(template, fillers):
    """Fill {placeholder} fields in a template with random choices."""
    import re
    result = template
    placeholders = re.findall(r'\{(\w+)\}', template)
    for ph in placeholders:
        if ph in fillers:
            choice = random.choice(fillers[ph])
            result = result.replace("{" + ph + "}", str(choice), 1)
    return result


# Generate 150 unique questions
all_questions = []
target_per_category = 150 // len(DAA_QUESTION_TEMPLATES) + 1

for category, templates in DAA_QUESTION_TEMPLATES.items():
    seen = set()
    attempts = 0
    while len([q for c, q in all_questions if c == category]) < target_per_category and attempts < 200:
        template = random.choice(templates)
        q = fill_template(template, FILLERS)
        if q not in seen:
            seen.add(q)
            all_questions.append((category, q))
        attempts += 1

# Trim to exactly 150
random.shuffle(all_questions)
all_questions = all_questions[:150]

print(f"Generated {len(all_questions)} questions across {len(set(c for c,q in all_questions))} categories")
print("\nSample (first 5):")
for c, q in all_questions[:5]:
    print(f"  [{c}] {q}")


## Step 3: Save the question stubs for use with Claude.ai

In [ ]:
questions_file = PROJECT_ROOT / "data" / "dpo_questions.json"
questions_data = {
    "description": "DAA question stubs for DPO preference pair generation",
    "questions": [{"id": i+1, "category": c, "question": q} for i, (c, q) in enumerate(all_questions)]
}
save_json(questions_data, "dpo_questions.json", results_dir="data")
print(f"Saved {len(all_questions)} questions to data/dpo_questions.json")


## Step 4: Generate batched prompts for Claude.ai

The cell below prints copy-paste-ready prompts. Each batch handles 10 questions.
You go to claude.ai, paste the system instruction once, then paste each batch.


In [ ]:
SYSTEM_INSTRUCTION = '''You are helping me build a preference dataset for fine-tuning a DAA (Design and Analysis of Algorithms) tutor. For each question I give you, write TWO responses:

A) CHOSEN: A Socratic guidance response - walks the student through the reasoning step by step, asks leading questions, explains intuition, builds up the answer gradually. Does NOT just dump the final answer. ~150-300 words.

B) REJECTED: A direct answer-dump response - gives the formula/answer immediately, minimal explanation, no guiding questions, treats the student as needing just the bottom line. ~50-150 words.

Format your response in strict JSON like this:
[
  {"id": 1, "chosen": "...", "rejected": "..."},
  {"id": 2, "chosen": "...", "rejected": "..."},
  ...
]

Output ONLY the JSON array, no other commentary.
'''

print("=" * 70)
print("STEP 1: Open https://claude.ai in a new tab")
print("STEP 2: Start a new chat, paste this system instruction:")
print("=" * 70)
print(SYSTEM_INSTRUCTION)

print("\n" + "=" * 70)
print("STEP 3: Paste each batch below as separate messages in the same chat.")
print("Save Claude's JSON response to data/dpo_responses_batch_N.json")
print("=" * 70)

BATCH_SIZE = 10
for batch_idx in range(0, len(all_questions), BATCH_SIZE):
    batch = all_questions[batch_idx:batch_idx + BATCH_SIZE]
    print(f"\n----- BATCH {batch_idx // BATCH_SIZE + 1} -----")
    print("Generate responses for these questions:")
    print()
    for i, (cat, q) in enumerate(batch):
        global_id = batch_idx + i + 1
        print(f'{{"id": {global_id}, "question": "{q}"}}')
    print()


## Step 5: Collect Claude's responses

After running each batch through Claude.ai, save each JSON response to a file in `data/`:
- `data/dpo_responses_batch_1.json`
- `data/dpo_responses_batch_2.json`
- ... etc

The cell below then loads them all and combines them into the final DPO dataset.


In [ ]:
# This cell combines all batch responses into the final dpo_pairs.json
import re

data_dir = PROJECT_ROOT / "data"
batch_files = sorted(data_dir.glob("dpo_responses_batch_*.json"))

if not batch_files:
    print("⚠ No batch response files found yet.")
    print("After getting responses from Claude.ai, save each batch's JSON output to:")
    print(f"  {data_dir}/dpo_responses_batch_1.json, ..._batch_2.json, etc.")
else:
    all_pairs = []
    questions_lookup = {i+1: q for i, (c, q) in enumerate(all_questions)}

    for bf in batch_files:
        with open(bf) as f:
            try:
                batch_data = json.load(f)
            except json.JSONDecodeError:
                # Try to extract JSON from the file if Claude added prose
                text = open(bf).read()
                m = re.search(r'\[.*\]', text, re.DOTALL)
                if m:
                    batch_data = json.loads(m.group())
                else:
                    print(f"Could not parse {bf}, skipping")
                    continue

        for item in batch_data:
            pid = item.get("id")
            if pid in questions_lookup:
                all_pairs.append({
                    "prompt": questions_lookup[pid],
                    "chosen": item["chosen"],
                    "rejected": item["rejected"]
                })

    print(f"Compiled {len(all_pairs)} preference pairs.")

    dpo_dataset = {
        "description": "DPO preference pairs for DAA tutor: chosen=Socratic guidance, rejected=answer dump",
        "n_pairs": len(all_pairs),
        "pairs": all_pairs
    }
    save_json(dpo_dataset, "dpo_pairs.json", results_dir="data")
    print(f"\n✓ Saved to data/dpo_pairs.json")
    print(f"\nSample pair:")
    if all_pairs:
        print(f"PROMPT: {all_pairs[0]['prompt']}")
        print(f"CHOSEN: {all_pairs[0]['chosen'][:200]}...")
        print(f"REJECTED: {all_pairs[0]['rejected'][:200]}...")
